In [1]:
# MIMIC-III Demo = 100 real ICU patients, openly available, no approval needed
# Contains REAL clinical notes — discharge summaries, nursing notes, doctor notes
# Perfect for testing our full pipeline before getting full MIMIC-III access

# ── Step 1: Download ──────────────────────────────────────────────────────────
# dont download it it is 46gb dataset use the some fils instead 
!wget -r -N -c -np \
      --user your_physionet_username \
      --password your_physionet_password \
      -P /home/zeus/content/mimic_demo/ \
      https://physionet.org/files/mimiciii-demo/1.4/

# Replace your_physionet_username and your_physionet_password
# with your PhysioNet account credentials (free to register)
# physionet.org/register — takes 2 minutes

# ── Step 2: Check what downloaded ────────────────────────────────────────────
!ls /home/zeus/content/mimic_demo/physionet.org/files/mimiciii-demo/1.4/

# You should see files like:
# NOTEEVENTS.csv.gz   ← THIS IS WHAT WE WANT (clinical notes)
# DIAGNOSES_ICD.csv.gz
# PATIENTS.csv.gz
# ADMISSIONS.csv.gz
# ... and more

--2026-04-14 05:05:39--  https://physionet.org/files/mimiciii-demo/1.4/
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘/home/zeus/content/mimic_demo/physionet.org/files/mimiciii-demo/1.4/index.html’

physionet.org/files     [ <=>                ]   3.50K  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-04-14 05:05:41 (265 MB/s) - ‘/home/zeus/content/mimic_demo/physionet.org/files/mimiciii-demo/1.4/index.html’ saved [3588]

Loading robots.txt; please ignore errors.
--2026-04-14 05:05:41--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 22 [text/plain]
Saving to: ‘/home/zeus/content/mimic_demo/physionet.org/robots.txt’

physionet.org/robot 100%[===================>]      22  --.-KB/s    in 

In [3]:
# Step 1: Install Kaggle API
!pip install kaggle -q

# Step 2: Set up your Kaggle credentials
# Go to kaggle.com → Your Profile → Settings → API → "Create New Token"
# This downloads a kaggle.json file to your computer
# Then paste your username and key below:

import os
os.environ["KAGGLE_USERNAME"] = "bhutiyalakhan"   # ← replace this
os.environ["KAGGLE_KEY"]      = "KGAT_bcb848d02dff564b313cedd31094b9d8"    # ← replace this

# Step 3: Download the dataset directly into Lightning AI
!kaggle competitions download -c histopathologic-cancer-detection -p /home/zeus/content/cancer_data/

# Step 4: Unzip it
!unzip -q /home/zeus/content/cancer_data/histopathologic-cancer-detection.zip \
          -d /home/zeus/content/cancer_data/

# Step 5: Confirm files are there
!ls /home/zeus/content/cancer_data/

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
unzip:  cannot find or open /home/zeus/content/cancer_data/histopathologic-cancer-detection.zip, /home/zeus/content/cancer_data/histopathologic-cancer-detection.zip.zip or /home/zeus/content/cancer_data/histopathologic-cancer-detection.zip.ZIP.
sample_submission.csv  test  train  train_labels.csv


In [1]:
# Install HuggingFace Transformers if not already installed
# transformers = the library that gives us BERT
# tqdm = progress bar so we can see generation progress

!pip install transformers tqdm -q

import os
import random
import pandas as pd
import numpy as np
import torch

from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# Check if GPU is available — BERT is slow on CPU for large datasets
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# You should see: Using device: cuda

Using device: cuda


In [9]:
import pandas as pd
import os

BASE = "/home/zeus/content/mimic_demo/physionet.org/files/mimiciii-demo/1.4/"

# ── Load clinical notes ───────────────────────────────────────────────────────
notes_df = pd.read_csv(os.path.join(BASE, "NOTEEVENTS.csv"), low_memory=False)

print(f"Total notes: {len(notes_df)}")
print(f"Categories available:\n{notes_df['category'].value_counts()}")

Total notes: 0
Categories available:
Series([], Name: count, dtype: int64)


In [10]:
# ── Keep discharge summaries — lowercase column names ─────────────────────────
discharge = notes_df[notes_df["category"] == "Discharge summary"].copy()
print(f"Discharge summaries: {len(discharge)}")

# ── Load diagnoses ────────────────────────────────────────────────────────────
diag_df = pd.read_csv(os.path.join(BASE, "DIAGNOSES_ICD.csv"), low_memory=False)

# Lowercase columns too
print(f"\nDiagnoses columns: {diag_df.columns.tolist()}")

# ICD-9 codes 140–208 = malignant cancers
def is_cancer(code):
    try:
        return 1 if 140 <= int(str(code)[:3]) <= 208 else 0
    except:
        return 0

diag_df["cancer_label"] = diag_df["icd9_code"].apply(is_cancer)   # ← lowercase

# One label per patient
labels = diag_df.groupby("subject_id")["cancer_label"].max().reset_index()
labels.columns = ["subject_id", "label"]

print(f"\nTotal patients  : {len(labels)}")
print(f"Cancer patients : {labels['label'].sum()}")
print(f"Normal patients : {(labels['label']==0).sum()}")

# ── Merge notes + labels ──────────────────────────────────────────────────────
merged = discharge.merge(labels, on="subject_id", how="inner")  # ← lowercase
merged = merged[["subject_id", "text", "label"]].dropna().reset_index(drop=True)

print(f"\nFinal paired records : {len(merged)}")
print("\n=== SAMPLE REAL NOTE (first 300 chars) ===")
print(merged["text"].iloc[0][:300])

Discharge summaries: 0

Diagnoses columns: ['row_id', 'subject_id', 'hadm_id', 'seq_num', 'icd9_code']

Total patients  : 100
Cancer patients : 29
Normal patients : 71

Final paired records : 0

=== SAMPLE REAL NOTE (first 300 chars) ===


IndexError: single positional indexer is out-of-bounds

In [ ]:
# Discharge summaries are the most information-rich notes
# They summarize the entire hospital stay — diagnoses, treatment, history
# Perfect for our cancer classification task

discharge_notes = notes_df[notes_df["CATEGORY"] == "Discharge summary"].copy()

print(f"Discharge summaries found: {len(discharge_notes)}")
print("\n=== SAMPLE REAL CLINICAL NOTE ===")
print(discharge_notes["TEXT"].iloc[0][:1000])  # first 1000 chars of first note

In [ ]:
# Load diagnoses to get cancer labels
# ICD-9 codes starting with 140-208 = malignant cancers

diag_path = os.path.join(MIMIC_PATH, "DIAGNOSES_ICD.csv.gz")
diag_df   = pd.read_csv(diag_path, compression="gz", low_memory=False)

# Flag cancer patients using ICD-9 codes
# 140–208 = malignant neoplasms (cancers)
def is_cancer_icd(code):
    try:
        # ICD-9 codes can be strings like "1400", "2080"
        num = int(str(code)[:3])
        return 1 if 140 <= num <= 208 else 0
    except:
        return 0

diag_df["cancer_label"] = diag_df["ICD9_CODE"].apply(is_cancer_icd)

# Get one label per patient (1 if ANY diagnosis is cancer)
patient_labels = diag_df.groupby("SUBJECT_ID")["cancer_label"].max().reset_index()
patient_labels.columns = ["SUBJECT_ID", "label"]

print(f"Total patients: {len(patient_labels)}")
print(f"Cancer patients : {patient_labels['label'].sum()}")
print(f"Normal patients : {(patient_labels['label']==0).sum()}")

# ── Merge notes with labels ───────────────────────────────────────────────────
merged = discharge_notes.merge(patient_labels, on="SUBJECT_ID", how="inner")
merged = merged[["SUBJECT_ID", "TEXT", "label"]].dropna()

print(f"\nFinal paired dataset: {len(merged)} notes with labels")
print(merged["label"].value_counts())

In [ ]:
# Now replace synthetic notes with REAL clinical notes
# Your BERT encoder code from Step 3 works exactly the same
# Just swap the text source!

real_notes  = merged["TEXT"].tolist()
real_labels = merged["label"].tolist()

print(f"Ready to encode {len(real_notes)} REAL clinical notes through BERT")
print("\n=== REAL NOTE PREVIEW ===")
print(real_notes[0][:500])

# Now run your existing extract_bert_embeddings() function:
# real_embeddings = extract_bert_embeddings(real_notes, tokenizer, bert_model)
#
# Everything else — fusion, training, Grad-CAM — stays exactly the same

In [ ]:
# MIMIC Demo only has ~100 patients
# Our image dataset has 220,025 images
# So we generate synthetic notes for images that don't have a real note
# Real notes are used where available, synthetic fills the rest

random.seed(42)

AGES_C = list(range(45, 85))
AGES_N = list(range(20, 65))

SMOKING = {
    1: ["30-pack-year smoking history",
        "active smoker, 1.5 packs/day",
        "former smoker, quit 5 years ago",
        "40-pack-year history, current smoker"],
    0: ["non-smoker", "never smoked",
        "quit smoking 15 years ago",
        "occasional social smoker"]
}
SYMPTOMS = {
    1: ["persistent cough for 3 months",
        "unexplained weight loss of 8 kg",
        "blood in sputum",
        "night sweats and fatigue",
        "enlarged lymph nodes"],
    0: ["routine annual checkup",
        "mild seasonal allergies",
        "no significant complaints",
        "routine pre-employment screening"]
}
FAMILY = {
    1: ["father diagnosed with lung cancer",
        "mother had breast cancer",
        "two siblings with colorectal cancer",
        "strong family history of cancer"],
    0: ["no family history of cancer",
        "family history unremarkable",
        "parents alive and healthy"]
}
LABS = {
    1: ["CEA elevated at 8.4 ng/mL",
        "CA-125 markedly elevated",
        "LDH elevated, suggestive of rapid cell turnover",
        "PSA 12.3 ng/mL, above normal range"],
    0: ["CEA within normal limits",
        "CBC unremarkable",
        "all tumor markers within normal range",
        "metabolic panel normal"]
}
BIOPSY = {
    1: ["Biopsy recommended based on imaging findings.",
        "Core biopsy reveals atypical cellular morphology.",
        "Fine needle aspiration performed — results pending."],
    0: ["No biopsy indicated at this time.",
        "No suspicious lesions detected.",
        "Routine monitoring advised."]
}

def generate_note(label):
    age  = random.choice(AGES_C if label == 1 else AGES_N)
    sex  = random.choice(["male", "female"])
    return (
        f"Patient is a {age}-year-old {sex} presenting for oncology evaluation. "
        f"Social history: {random.choice(SMOKING[label])}. "
        f"Chief complaint: {rando

In [5]:
import os

# Step 1: Check the base mimic_demo folder structure
print("=== TOP LEVEL ===")
for root, dirs, files in os.walk("/home/zeus/content/mimic_demo/"):
    level = root.replace("/home/zeus/content/mimic_demo/", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

=== TOP LEVEL ===
/
physionet.org/
  robots.txt
  files/
    mimiciii-demo/
      1.4/
        SERVICES.csv
        ICUSTAYS.csv
        D_CPT.csv
        CALLOUT.csv
        D_ITEMS.csv
        INPUTEVENTS_CV.csv
        TRANSFERS.csv
        MICROBIOLOGYEVENTS.csv
        PATIENTS.csv
        DIAGNOSES_ICD.csv
        CAREGIVERS.csv
        ADMISSIONS.csv
        D_LABITEMS.csv
        PRESCRIPTIONS.csv
        PROCEDUREEVENTS_MV.csv
        CPTEVENTS.csv
        index.html
        D_ICD_PROCEDURES.csv
        NOTEEVENTS.csv
        LABEVENTS.csv
        CHARTEVENTS.csv
        LICENSE.txt
        INPUTEVENTS_MV.csv
        D_ICD_DIAGNOSES.csv
        DRGCODES.csv
        SHA256SUMS.txt
        OUTPUTEVENTS.csv
        PROCEDURES_ICD.csv
        DATETIMEEVENTS.csv


In [8]:
print(notes_df.columns.tolist())
print("\n")
print(notes_df.head(2))

['row_id', 'subject_id', 'hadm_id', 'chartdate', 'charttime', 'storetime', 'category', 'description', 'cgid', 'iserror', 'text']


Empty DataFrame
Columns: [row_id, subject_id, hadm_id, chartdate, charttime, storetime, category, description, cgid, iserror, text]
Index: []


In [11]:
# Check exact category names in your file
print(notes_df['category'].unique())
print("\nValue counts:")
print(notes_df['category'].value_counts())

[]

Value counts:
Series([], Name: count, dtype: int64)


In [12]:
import pandas as pd
import os

BASE = "/home/zeus/content/mimic_demo/physionet.org/files/mimiciii-demo/1.4/"

# Check every file and see which ones have data
for filename in sorted(os.listdir(BASE)):
    if filename.endswith(".csv"):
        try:
            df = pd.read_csv(os.path.join(BASE, filename), low_memory=False)
            print(f"{filename:40s} → {len(df):>6} rows, {len(df.columns)} cols")
        except Exception as e:
            print(f"{filename:40s} → ERROR: {e}")

ADMISSIONS.csv                           →    129 rows, 19 cols
CALLOUT.csv                              →     77 rows, 24 cols
CAREGIVERS.csv                           →   7567 rows, 4 cols
CHARTEVENTS.csv                          → 758355 rows, 15 cols
CPTEVENTS.csv                            →   1579 rows, 12 cols
DATETIMEEVENTS.csv                       →  15551 rows, 14 cols
DIAGNOSES_ICD.csv                        →   1761 rows, 5 cols
DRGCODES.csv                             →    297 rows, 8 cols
D_CPT.csv                                →    134 rows, 9 cols
D_ICD_DIAGNOSES.csv                      →  14567 rows, 4 cols
D_ICD_PROCEDURES.csv                     →   3882 rows, 4 cols
D_ITEMS.csv                              →  12487 rows, 10 cols
D_LABITEMS.csv                           →    753 rows, 6 cols
ICUSTAYS.csv                             →    136 rows, 12 cols
INPUTEVENTS_CV.csv                       →  34799 rows, 22 cols
INPUTEVENTS_MV.csv                       →  132